# Distill SFT (Colab, free T4) + 2x2 eval
Runtime → GPU (T4). Put `brickexp_code.zip` in `MyDrive/brickexp_distill/` (or upload it to /content). Run all; after a disconnect just Run all again — finished stages are skipped. Checker = connectivity (same as C1); Gurobi re-judge happens locally.

In [ ]:
import os, sys, glob, json, time, zipfile, subprocess, traceback
from google.colab import drive
drive.mount("/content/drive")
OUTD = "/content/drive/MyDrive/brickexp_distill"
os.makedirs(OUTD, exist_ok=True); T0 = time.time()
def stage(n):
    print(f"=== [{(time.time()-T0)/60:6.1f} min] {n}", flush=True)
    open(f"{OUTD}/stage_{int(time.time()-T0):06d}_{n.replace(' ', '_')}.txt", "w").write(time.ctime())
def run_logged(cmd, tag):
    stage(tag)
    p = subprocess.run(f"set -o pipefail; ( {cmd} ) 2>&1 | tee {OUTD}/{tag}.log", shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        open(f"{OUTD}/error.txt", "a").write(f"[{tag}] rc={p.returncode}\n" + p.stdout[-3000:])
        raise RuntimeError(f"{tag} failed rc={p.returncode}")
    return p
stage("pip")
r = subprocess.run("pip install -q vllm peft gurobipy datasets networkx bitsandbytes", shell=True, capture_output=True, text=True)
print(r.returncode, r.stderr[-1500:])
subprocess.run("pip uninstall -y -q torchao", shell=True)  # old torchao on Kaggle breaks recent peft import
print(subprocess.run("pip list 2>/dev/null | grep -iE '^(vllm|torch|transformers|peft) '", shell=True, capture_output=True, text=True).stdout)

In [ ]:
WORK = "/content/RLtheLEGO"  # code on local disk, outputs on Drive
zips = glob.glob(OUTD + "/brickexp_code.zip") + glob.glob("/content/**/brickexp_code.zip", recursive=True)
if not os.path.exists(WORK + "/brickexp"):
    assert zips, "attach the dataset containing brickexp_code.zip"
    zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# no secrets on Colab: checker = connectivity (as in the C1 eval); stability re-judged locally later
from brickexp import checker
print("Gurobi available here:", checker.gurobi_available())
n = sum(1 for _ in open("data/distill/sft_distill.jsonl")); print("SFT examples:", n); assert n > 0

In [ ]:
DONE_TRAIN = os.path.exists(f"{OUTD}/sft/train_log.json") and (os.path.exists(f"{OUTD}/sft/merged/config.json") or os.path.exists(f"{OUTD}/runs/summary_sft.json"))
if DONE_TRAIN:
    print('train: already done on Drive, skipping')
else:
    # ---- LoRA SFT, fp16 on one T4 (subprocess so the GPU is free for vLLM afterwards) ----
    TRAIN = (f"export CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; "
             f"python -m brickexp.train_sft --model Qwen/Qwen2.5-1.5B-Instruct --data_jsonl data/distill/sft_distill.jsonl --out {OUTD}/sft "
             f"--epochs 2 --bs 2 --grad_accum 8 --max_len 3072 --rank 16 --lr 2e-4 --max_steps -1")
    pr = run_logged(TRAIN + " --probe_steps 5 --no_merge", "train_probe")   # memory probe before the full run
    import re; m = re.search(r"PEAK_MEM_GB ([0-9.]+)", pr.stdout); PEAK = float(m.group(1)) if m else None
    print("probe peak GB", PEAK); assert PEAK is not None and PEAK < 13.5, "would OOM on T4 (15 GB)"
    run_logged(TRAIN, "train_full")
    subprocess.run(f"rm -rf {OUTD}/sft/ckpt", shell=True)
    L = json.load(open(f"{OUTD}/sft/train_log.json"))
    print("loss:", [(h.get("step"), round(h["loss"], 4)) for h in L["log_history"] if "loss" in h])

In [ ]:
if os.path.exists(f"{OUTD}/runs/summary_base.json"):
    print('eval base: done, skipping')
else:
    run_logged(f"python -m brickexp.eval_2x2 run --backend vllm --max_budget 3 --checker connectivity --limit 0 --time_budget_h 3.0 --out_dir {OUTD}/runs --model Qwen/Qwen2.5-1.5B-Instruct --tag base", "eval_base")

In [ ]:
if os.path.exists(f"{OUTD}/runs/summary_sft.json"):
    print('eval sft: done, skipping')
else:
    run_logged(f"python -m brickexp.eval_2x2 run --backend vllm --max_budget 3 --checker connectivity --limit 0 --time_budget_h 3.0 --out_dir {OUTD}/runs --model {OUTD}/sft/merged --tag sft", "eval_sft")

In [ ]:
S = " ".join(f"{OUTD}/runs/summary_{t}.json" for t in ['base', 'sft'])
print(run_logged(f"python -m brickexp.eval_2x2 table {S} --out {OUTD}/TABLE_2x2.md", "table").stdout)
subprocess.run(f"rm -rf {OUTD}/sft/merged", shell=True)  # keep adapter + logs only
stage("done")